# RNA desirability calibration -- L (floor) and T (target)

**Why we're doing this.** `scoring/desirability.py`'s Derringer & Suich (1980) transform needs a floor `L` and a target `T`, in `log2(TPM+1)`, for every gene, before it can turn a raw RNA measurement into a desirability `d` in `[0, 1]`. The paper itself gives no data-driven procedure for these constants (its own Gaps section says so) -- this calibration is this project's own extension, and it is written down here so the choice is checkable, not buried in code.

**What we compute:**
1. A **global** `(L, T)` per gene: the 10th/90th percentile of `log2(TPM+1)` across every measured line in `data/processed/expression_rna.csv`.
2. A **per-lineage override**: the same percentiles computed *within* a lineage, for any (gene, lineage) pair with at least 15 measured lines -- the same minimum-group-size threshold `docs/plan/CONSTRAINTS.md` SS3 already uses elsewhere in this project for lineage-stratified statistics. Below that count we fall back to the global value, because a percentile computed on fewer than 15 lines is not a stable estimate.

**Why a per-lineage override at all?** A gene that is legitimately silent in one lineage and highly expressed in another should not be judged against one shared threshold -- scoring a gene against a pan-cancer floor would penalise a line simply for being the 'wrong' tissue, not for lacking the biology the query actually asked about.

**A correction from the earlier (v4) design, stated plainly.** The original version of this calibration used Jin et al. (2023)'s external RNA panel for the per-lineage override. That file (`data/augmented/validation/jin2023_supplementary_data5.xlsx`) is explicitly **reserved for external validation** under this branch's data policy (root `_.md` SS10; `preprocessing/preprocess.py` never reads it) -- using it here would leak the held-out validation set into a scoring constant. The per-lineage override is instead derived **in-sample**, from the same `expression_rna.csv` the global default already comes from, just stratified by `cell_lines.lineage`. See `docs/reference/ALGORITHM_SPEC.md`'s 'Where L and T come from' section for the full writeup.

**The computation lives in `scoring/build_desirability_constants.py`, not in this notebook.** That script is what actually ships the constants (`python scoring/build_desirability_constants.py`, run by hand from the repo root). This notebook imports the very same functions and walks through them step by step, so the audit you are reading and the file the scorer loads can never drift apart. Before this split the two were separate copies of the same percentile code, and they *had* drifted -- see SS7.

**Output:** `scoring/resources/desirability_constants.json` and `scoring/resources/desirability_constants_rna_sweep_floor.json`. Both are regenerable from `data/raw/` + `data/augmented/` alone, per root `_.md` SS4's reproducibility contract.

In [ ]:
import json
import sys
import time

sys.path.insert(0, "..")  # this notebook runs from scoring/notebooks/; the builder is one level up

import build_desirability_constants as build

DATA_DIR = "../../data/processed"
RESOURCES_DIR = "../resources"

MIN_LINEAGE_N = build.MIN_LINEAGE_N              # 15 -- CONSTRAINTS.md SS3's lineage-size threshold
MIN_LINEAGE_N_FLOOR = build.MIN_LINEAGE_N_FLOOR  # 5 -- below V6-3's swept range, SENSITIVITY.md SS3
print(f"production MIN_LINEAGE_N={MIN_LINEAGE_N}, sweep floor={MIN_LINEAGE_N_FLOOR}")

## 1. Load the RNA table and the lineage spine

`expression_rna.csv` is ~79 million rows -- we read it with narrow dtypes (`category` for the two ID columns, `float32` for the value) so it fits comfortably in memory. `cell_lines.csv` gives us the lineage for every `ModelID`.

In [ ]:
t0 = time.time()
expr = build.load_expression_with_lineage(DATA_DIR)
print(f"{len(expr):,} rows in {time.time() - t0:.1f}s")
expr.head()

## 2. Global L/T -- 10th/90th percentile per gene, pooled across every measured line

In [ ]:
global_q = build.compute_global_lt(expr)
print(f"{len(global_q):,} genes have a global RNA measurement")
global_q.head()

## 3. Per-lineage L/T -- same percentiles, computed within `lineage`

`compute_lineage_lt` returns the percentiles for **every** (lineage, gene) pair together with each pair's measured-line count, and applies **no count floor of its own**. That separation is deliberate: the production file and the sweep-floor file want the same percentiles cut at different thresholds, so the cut belongs to the caller, not to the statistic.

A gene/lineage combination measured in fewer than 15 lines does not get its own override in the production file -- `desirability.get_lt` falls back to the global value for it.

In [ ]:
lineage_q, lineage_counts = build.compute_lineage_lt(expr)
print(f"{len(lineage_q):,} (lineage, gene) pairs measured at all")
print(f"{(lineage_counts >= MIN_LINEAGE_N).sum():,} of them have >= {MIN_LINEAGE_N} measured lines")
lineage_q.head()

## 4. The discrimination gate -- drop any (gene[, lineage]) where T is not strictly greater than L

If a gene reads the same value at its 10th and 90th percentile, there is no discrimination available and `desirability_transform` would divide by zero -- these genes are dropped from the constants file, not defaulted to an arbitrary range.

In [ ]:
global_kept = build.apply_discrimination_gate(global_q)
lineage_kept = build.apply_discrimination_gate(lineage_q)
print(f"Global: kept {len(global_kept):,} of {len(global_q):,} genes")
print(f"Per-lineage: kept {len(lineage_kept):,} of {len(lineage_q):,} pairs")

## 5. Write `scoring/resources/desirability_constants.json`

The count floor is applied here, at the cut, not upstream in SS3 -- `to_per_lineage_dict` keeps only pairs measured in at least `MIN_LINEAGE_N` lines.

In [ ]:
per_lineage_dict, _ = build.to_per_lineage_dict(lineage_kept, lineage_counts, MIN_LINEAGE_N)
global_dict = build.to_global_dict(global_kept)

constants = build.build_production_constants(global_q, global_kept, per_lineage_dict)

out_path = f"{RESOURCES_DIR}/desirability_constants.json"
with open(out_path, "w") as f:
    json.dump(constants, f, indent=2)
print(f"Wrote {out_path}")
print(f"{sum(len(v) for v in per_lineage_dict.values()):,} (lineage, gene) overrides "
      f"across {len(per_lineage_dict)} lineages")

## 6. Sanity check -- a couple of well-known genes

EGFR and GAPDH should both show a wide floor-to-target range (they are commonly expressed and variable across the panel); a housekeeping gene like GAPDH should sit at a high floor almost everywhere.

In [7]:
for symbol, ensembl_id in [("EGFR", "ENSG00000146648"), ("GAPDH", "ENSG00000111640")]:
    if ensembl_id in global_dict:
        print(symbol, global_dict[ensembl_id])
    else:
        print(symbol, "not in the kept gene set")

EGFR {'L': 0.111, 'T': 6.2072}
GAPDH {'L': 11.09, 'T': 12.7872}


## 7. V6-3 sweep-floor export -- `scoring/resources/desirability_constants_rna_sweep_floor.json`

`docs/plan/PARAMETERS.md` §11 requires the sensitivity sweep to perturb `min_lineage_n` (default 15,
row 4, tag T1) over a range as low as 5. The production file above only keeps (lineage, gene) pairs
that already clear 15 measured lines -- pairs below that are **never persisted**, so a sweep sample
asking "what if the floor were 5?" cannot be answered from the production file alone.

No new statistics are needed for this, only a **lower cut** of the same `lineage_q`/`lineage_counts`
computed in §3 -- at `MIN_LINEAGE_N_FLOOR=5`, below every value V6-3's swept range for T1 (`[5, 50]`,
`docs/plan/SENSITIVITY.md` §3) will ever ask for. The export additionally persists the
per-(lineage, gene) sample counts (`lineage_gene_counts`), which the production file does not carry.
At sweep time `scoring/sensitivity.py::build_rna_constants_at_threshold` filters this file down to
`count >= sampled_threshold` -- a dict comprehension, not a recomputation.

> **Defect fixed 2026-08-25 — the earlier version of this export was inert.** When this notebook
> owned its own copy of the percentile code, §3 rebound `lineage_q` to the `>= 15` subset *before*
> this section read it. The floor export therefore re-cut an already-15-gated frame at 5, which
> removes nothing, and the notebook's own recorded output said so plainly: `929,610 (lineage, gene)
> overrides at floor=5 vs. 929,610 at production MIN_LINEAGE_N=15`, identical. The shipped
> sweep-floor file was a byte-for-byte copy of the production per-lineage block plus counts, so
> every T1 sample in `[5, 15]` re-gated to exactly the production constants and moved nothing.
> `compute_lineage_lt` now applies no floor at all and the cut happens per-caller (§3, §5), which is
> what makes the two files genuinely differ. **V6-3's reported T1 sensitivity predates this fix and
> under-states T1's true influence** — see `docs/plan/SENSITIVITY.md` §10.

**This file is not committed** -- at floor=5 it runs to ~100MB (nearly every (lineage, gene) pair in
the panel clears a floor that low), which would break this repo's data-hygiene rule (root `_.md` §11)
if committed directly. It regenerates from `data/raw/` + `data/augmented/` by re-running
`scoring/build_desirability_constants.py`, the same reproducibility contract every other generated
calibration artefact in this project already follows -- V6-3's own seed (recorded in
`docs/plan/SENSITIVITY.md`) is what makes the *sweep itself* reproducible, not a committed copy of
every intermediate calibration file.

In [ ]:
per_lineage_floor, lineage_gene_counts = build.to_per_lineage_dict(
    lineage_kept, lineage_counts, MIN_LINEAGE_N_FLOOR
)

sweep_floor_constants = build.build_sweep_floor_constants(
    global_q, global_kept, per_lineage_floor, lineage_gene_counts
)

floor_out_path = f"{RESOURCES_DIR}/desirability_constants_rna_sweep_floor.json"
with open(floor_out_path, "w") as f:
    json.dump(sweep_floor_constants, f, indent=2)
print(f"Wrote {floor_out_path}")

n_floor = sum(len(v) for v in per_lineage_floor.values())
n_production = sum(len(v) for v in per_lineage_dict.values())
print(f"{n_floor:,} (lineage, gene) overrides at floor={MIN_LINEAGE_N_FLOOR} "
      f"vs. {n_production:,} at production MIN_LINEAGE_N={MIN_LINEAGE_N}")
# the whole point of the floor export -- if these two are equal, the sweep has nothing to sweep
assert n_floor > n_production, "floor export is inert -- see the defect note above"